# SWARM: Job Selection via Consensus - Multi Site

## Import the libraries

In [1]:
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
import ipaddress

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager(project_id="3a05ccb3-a4b9-4bc8-9bc8-4c8eb65c9d3e")
                     
fablib.show_config();

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
Token File,/Users/kthare10/work/fabric_config/id_token.json
Project ID,3a05ccb3-a4b9-4bc8-9bc8-4c8eb65c9d3e
Bastion Host,bastion.fabric-testbed.net
Bastion Username,kthare10_0011904101
Bastion Private Key File,/Users/kthare10/.ssh/bastion-prod-2
Slice Public Key File,/Users/kthare10/.ssh/id_rsa.pub
Slice Private Key File,/Users/kthare10/.ssh/id_rsa


## Define variables

In [2]:
name_prefix = "agent"
agents_per_node = 1

# ============================================================
# TWO-SLICE SPLIT
# Toggle this to 1 or 2, then run the Slice-Creation cell.
# Provision & bring up SLICE_PART = 1 FIRST (it holds the database),
# then set SLICE_PART = 2 and run the creation cell again.
# ============================================================
SLICE_PART = 1

total_agents = 100
base_slice_name = "SWARM-MULTI"

# Per-part slice names (both live at the same time)
slice_name_1 = f"{base_slice_name}-{total_agents}-p1"
slice_name_2 = f"{base_slice_name}-{total_agents}-p2"
slice_name = slice_name_1 if SLICE_PART == 1 else slice_name_2

db_node_name = "database"

# Node profile parameters
cores = 16
ram = 16
disk = 500
image = "docker_ubuntu_22"
branch = "agent-recovery"
network_name = "fabv4"
# ============================================================
# MONITORING
# Every agent + the database get a second NIC (nic2) attached to a
# per-site FABNetv4 monitoring network. A separate "monitor" VM
# (slice 1) runs Prometheus + Grafana and scrapes node_exporter on
# every node over those networks.
# ============================================================
monitor_node_name = "monitor"
monitor_cores = 4
monitor_ram = 16
monitor_disk = 100           # disk is the binding placement constraint; keep small
mon_network_name = "fabv4mon"   # per-site monitoring networks (nic2)


## Determine sites

In [3]:
# Capacity-aware per-site agent counts (NOT an even split), set from LIVE
# per-host capacity honoring the 16c / 16g / 500d profile. The 500 GB disk is
# the binding constraint on most hosts, so site-level core totals badly
# overstate real placement room. Values drift fast under contention --
# ALWAYS re-verify per-host capacity immediately before submitting.
#
# Live real-max snapshot (16c/16g/500d): RUTG 15, PSC 12, GATECH 5, UTAH 5,
# GPN 5. Counts below leave headroom under those ceilings.
# The database lives in SLICE 1 and is counted separately (+1 VM on RUTG).
# NOTE: at 500 GB disk these sum to ~78 agents, not 100 -- the testbed does
# not currently have room for 100x500GB. To reach 100, drop `disk` to ~100 GB
# (see node profile cell) or add capacity. Excluded: EDUKY, EDC, BRIST, MAX.

plan_part1 = {
    "RUTG":   9,   # + database (14 VMs on RUTG; live real max 15)
    "PSC":    9,
    "GATECH":  4,
    "INDI":    2,
    "GPN":     4,
    "FIU":    7,
    "MASS":    8,
    "UTAH":    2
}   # = 35 agents

plan_part2 = {
    "CLEM":    9,
    "HAWI":    8,
    "WASH":    5,
    "AMST":     7,
    "KANS":    5,
    "MICH":    4,
    "PRIN":    3,
    "TACC":    3,
    "SRI":     3,
    #"CERN":    3
}   # = 50 agents

site_plan = plan_part1 if SLICE_PART == 1 else plan_part2
sites = list(site_plan.keys())

# Database placed on the roomiest site, which must live in slice 1.
db_site = "RUTG"

# Monitor VM (Prometheus/Grafana) also lives in slice 1. Its site must be
# in plan_part1 so its monitoring network exists. NOTE: the monitor adds
# one more (small, 100 GB) VM on this site -- verify live capacity.
monitor_site = db_site
assert monitor_site in plan_part1, "monitor_site must be in slice 1's plan"

# Sanity checks (total is whatever the plans sum to; not forced to 100)
total_planned = sum(plan_part1.values()) + sum(plan_part2.values())
if SLICE_PART == 1:
    assert db_site in plan_part1, "db_site must be in slice 1's plan"

n_here = sum(site_plan.values())
extra = " + 1 database" if SLICE_PART == 1 else ""
print(f'Slice part {SLICE_PART}: "{slice_name}" -> {n_here} agents{extra} across {sites}')
print(f'TOTAL planned agents across both slices: {total_planned}')

Slice part 1: "SWARM-MULTI-100-p1" -> 45 agents + 1 database across ['RUTG', 'PSC', 'GATECH', 'INDI', 'GPN', 'FIU', 'MASS', 'UTAH']
TOTAL planned agents across both slices: 92


## Slice Creation

- **Database Node**
  - Allocate a node to host the Redis database. Ensure this node is connected to the L3 FabNetV4 network to enable communication with the agent nodes.

- **Agent Cluster**
  - Provision the number of nodes specified by `swarm_node_count` for deploying Swarm agents, ideally distributing them across multiple sites.
  - Each agent node should also be connected to the L3 FabNetV4 network to facilitate inter-node communication.

In [ ]:
# Create Slice (run once per SLICE_PART)
slice = fablib.new_slice(name=slice_name)

agent_idx_start = 1 if SLICE_PART == 1 else sum(plan_part1.values()) + 1
agent_idx = agent_idx_start

# Database + monitor interface handles (only created in slice 1)
db_iface = None
db_mon_iface = None
mon_vm_iface = None
if SLICE_PART == 1:
    database = slice.add_node(name=db_node_name, site=db_site,
                              image=image, disk=disk, cores=cores, ram=ram)
    db_iface = database.add_component(model="NIC_Basic", name="nic1").get_interfaces()[0]
    db_mon_iface = database.add_component(model="NIC_Basic", name="nic2").get_interfaces()[0]

    # Separate monitor VM (Prometheus + Grafana). Its single NIC joins the
    # monitoring network of its site; FABNetv4 routes it to every other
    # site's monitoring network in both slices.
    monitor = slice.add_node(name=monitor_node_name, site=monitor_site,
                             image=image, disk=monitor_disk,
                             cores=monitor_cores, ram=monitor_ram)
    mon_vm_iface = monitor.add_component(model="NIC_Basic", name="nic1").get_interfaces()[0]

for site, count_here in site_plan.items():
    # One site-scoped FABNetv4 L3 network per site (primary, nic1).
    # All these networks live in the routable 10.128.0.0/10 space, so nodes
    # across BOTH slices can reach each other (incl. the shared database).
    net = slice.add_l3network(name=f"{network_name}-{site}", type="IPv4")
    # A second site-scoped FABNetv4 L3 network for monitoring traffic (nic2).
    mon_net = slice.add_l3network(name=f"{mon_network_name}-{site}", type="IPv4")
    print(f"Creating {count_here} agents for site: {site}")

    # Attach the database to its own site's networks (slice 1 only)
    if SLICE_PART == 1 and site == db_site:
        net.add_interface(db_iface)
        db_iface.set_mode("manual")
        mon_net.add_interface(db_mon_iface)
        db_mon_iface.set_mode("manual")

    # Attach the monitor VM to its site's monitoring network (slice 1 only)
    if SLICE_PART == 1 and site == monitor_site:
        mon_net.add_interface(mon_vm_iface)
        mon_vm_iface.set_mode("manual")

    for _ in range(count_here):
        agent = slice.add_node(
            name=f"{name_prefix}-{agent_idx}",
            site=site, image=image, disk=disk, cores=cores, ram=ram
        )
        agent_idx += 1
        iface = agent.add_component(model="NIC_Basic", name="nic1").get_interfaces()[0]
        iface.set_mode("manual")
        net.add_interface(iface)
        # Second NIC -> monitoring network
        mon_iface = agent.add_component(model="NIC_Basic", name="nic2").get_interfaces()[0]
        mon_iface.set_mode("manual")
        mon_net.add_interface(mon_iface)

# Submit Slice Request
slice.submit(wait=False)


In [ ]:
# Wait for the current SLICE_PART to come up.
# 50-ish VMs across several sites can take a while; bump timeout as needed.
slice = fablib.get_slice(slice_name)
#slice.wait(timeout=2400)
slice.wait_ssh()

In [ ]:
slice = fablib.get_slice(slice_name)
slice.post_boot_config()

In [4]:
slice = fablib.get_slice(slice_name)
slice.list_nodes();

User: kthare10@email.unc.edu bastion key is valid!
Configuration is valid


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
7cd6ddbf-9e78-488f-b1e2-24ccd657ba5a,agent-1,16,16,500,docker_ubuntu_22,qcow2,rutg-w1.fabric-testbed.net,RUTG,ubuntu,2620:0:d61:4101:f816:3eff:fe26:ce04,Active,,ssh -i /Users/kthare10/.ssh/id_rsa -F /Users/kthare10/work/fabric_config/ssh_config ubuntu@2620:0:d61:4101:f816:3eff:fe26:ce04,/Users/kthare10/.ssh/id_rsa.pub,/Users/kthare10/.ssh/id_rsa
72807abf-3761-4bca-862d-18e9710858b1,agent-10,16,16,500,docker_ubuntu_22,qcow2,psc-w3.fabric-testbed.net,PSC,ubuntu,2001:5e8:ff00:ffff:f816:3eff:fe78:eda0,Active,,ssh -i /Users/kthare10/.ssh/id_rsa -F /Users/kthare10/work/fabric_config/ssh_config ubuntu@2001:5e8:ff00:ffff:f816:3eff:fe78:eda0,/Users/kthare10/.ssh/id_rsa.pub,/Users/kthare10/.ssh/id_rsa
06f51946-e364-429b-b0d2-a2608a39abfa,agent-11,16,16,500,docker_ubuntu_22,qcow2,psc-w3.fabric-testbed.net,PSC,ubuntu,2001:5e8:ff00:ffff:f816:3eff:fe5c:6496,Active,,ssh -i /Users/kthare10/.ssh/id_rsa -F /Users/kthare10/work/fabric_config/ssh_config ubuntu@2001:5e8:ff00:ffff:f816:3eff:fe5c:6496,/Users/kthare10/.ssh/id_rsa.pub,/Users/kthare10/.ssh/id_rsa
9e44c055-5eea-4a2f-92c0-3fb492900964,agent-12,16,16,500,docker_ubuntu_22,qcow2,psc-w1.fabric-testbed.net,PSC,ubuntu,2001:5e8:ff00:ffff:f816:3eff:fe44:e75a,Active,,ssh -i /Users/kthare10/.ssh/id_rsa -F /Users/kthare10/work/fabric_config/ssh_config ubuntu@2001:5e8:ff00:ffff:f816:3eff:fe44:e75a,/Users/kthare10/.ssh/id_rsa.pub,/Users/kthare10/.ssh/id_rsa
1707f3d4-7e4e-4516-aa0e-ee0f997b879f,agent-13,16,16,500,docker_ubuntu_22,qcow2,psc-w1.fabric-testbed.net,PSC,ubuntu,2001:5e8:ff00:ffff:f816:3eff:fe20:23e4,Active,,ssh -i /Users/kthare10/.ssh/id_rsa -F /Users/kthare10/work/fabric_config/ssh_config ubuntu@2001:5e8:ff00:ffff:f816:3eff:fe20:23e4,/Users/kthare10/.ssh/id_rsa.pub,/Users/kthare10/.ssh/id_rsa
93faf0d5-be51-4c64-be05-b2bed9d52176,agent-14,16,16,500,docker_ubuntu_22,qcow2,psc-w1.fabric-testbed.net,PSC,ubuntu,2001:5e8:ff00:ffff:f816:3eff:fee7:a438,Active,,ssh -i /Users/kthare10/.ssh/id_rsa -F /Users/kthare10/work/fabric_config/ssh_config ubuntu@2001:5e8:ff00:ffff:f816:3eff:fee7:a438,/Users/kthare10/.ssh/id_rsa.pub,/Users/kthare10/.ssh/id_rsa
2ab98f2d-769e-4b3e-aa30-7e34615e731a,agent-15,16,16,500,docker_ubuntu_22,qcow2,psc-w1.fabric-testbed.net,PSC,ubuntu,2001:5e8:ff00:ffff:f816:3eff:fed0:5948,Active,,ssh -i /Users/kthare10/.ssh/id_rsa -F /Users/kthare10/work/fabric_config/ssh_config ubuntu@2001:5e8:ff00:ffff:f816:3eff:fed0:5948,/Users/kthare10/.ssh/id_rsa.pub,/Users/kthare10/.ssh/id_rsa
c501a45a-247c-45dd-81d6-76265f8ce957,agent-16,16,16,500,docker_ubuntu_22,qcow2,psc-w1.fabric-testbed.net,PSC,ubuntu,2001:5e8:ff00:ffff:f816:3eff:fe3a:923a,Active,,ssh -i /Users/kthare10/.ssh/id_rsa -F /Users/kthare10/work/fabric_config/ssh_config ubuntu@2001:5e8:ff00:ffff:f816:3eff:fe3a:923a,/Users/kthare10/.ssh/id_rsa.pub,/Users/kthare10/.ssh/id_rsa
05ccd3a1-d384-443d-bbb2-7430d1df8df7,agent-17,16,16,500,docker_ubuntu_22,qcow2,psc-w1.fabric-testbed.net,PSC,ubuntu,2001:5e8:ff00:ffff:f816:3eff:fe2a:1f50,Active,,ssh -i /Users/kthare10/.ssh/id_rsa -F /Users/kthare10/work/fabric_config/ssh_config ubuntu@2001:5e8:ff00:ffff:f816:3eff:fe2a:1f50,/Users/kthare10/.ssh/id_rsa.pub,/Users/kthare10/.ssh/id_rsa
b4bbb640-e412-499e-8238-75c55f698d4a,agent-18,16,16,500,docker_ubuntu_22,qcow2,psc-w1.fabric-testbed.net,PSC,ubuntu,2001:5e8:ff00:ffff:f816:3eff:fe1c:2465,Active,,ssh -i /Users/kthare10/.ssh/id_rsa -F /Users/kthare10/work/fabric_config/ssh_config ubuntu@2001:5e8:ff00:ffff:f816:3eff:fe1c:2465,/Users/kthare10/.ssh/id_rsa.pub,/Users/kthare10/.ssh/id_rsa


In [ ]:
slice.list_networks();

## Configure the fleet with `db_node_setup`

The cells below replace the old per-node `fablib` `upload_directory()` /
`execute()` loops. Those drive one SSH session per node from this machine and
time out well before a ~90 node slice is configured.

Instead, `db_node_setup/` does the same work with its own SSH transport:
bastion-aware (bracketing IPv6 management addresses and failing over between
FABRIC's bastions), multiplexed, parallel, logged per node, and idempotent.

`fablib` is still used above for slice creation and status -- that is control
plane only and is not affected by the SSH problem.

Everything here runs from **wherever this notebook runs**. Nothing needs to be
copied to the database node by hand.


In [ ]:
# Build the deployment plan from the FABRIC control plane only (no node SSH).
# Writes db_node_setup/plan/: management IPs, NIC MACs with their assigned
# FABNetv4 addresses, the /etc/hosts block and prometheus.yml.
import subprocess, sys

SETUP = "db_node_setup"

rc = subprocess.run([
    sys.executable, f"{SETUP}/gen_inventory.py",
    "--slices", slice_name_1, slice_name_2,
    "--db-node", db_node_name,
    "--monitor-node", monitor_node_name,
    "--branch", branch,
    "--network-prefix", network_name,
    "--mon-network-prefix", mon_network_name,
]).returncode
assert rc == 0, "gen_inventory.py failed -- are both slice parts up?"


In [ ]:
# Check every node is reachable before doing any work.
# Prints the transport chosen per node (direct, or which bastion).
!bash db_node_setup/00_check_access.sh


### Run the setup pipeline

`setup_all.sh` runs the steps below in order. Each is idempotent, so a failed
run can simply be repeated; pass a step number to resume (`setup_all.sh 3`).

| Step | Does |
|------|------|
| 0 | SSH reachability probe, caching the working transport per node |
| 1 | netplan on both dataplane NICs, interface resolved by MAC on the node |
| 2 | root SSH mesh across all nodes |
| 3 | `/etc/hosts` block |
| 4 | clone + fan out SwarmAgents to every agent |
| 5 | Python dependencies |
| 6 | node_exporter + Prometheus/Grafana (skipped if the sources are absent) |

Per-node logs land in `db_node_setup/logs/`. This takes a while at ~90 nodes --
step 1 is the slowest.


In [ ]:
!bash db_node_setup/setup_all.sh


## Running SWARM-MULTI Consensus Setup

`run_on.sh` is the fablib-free replacement for `node.execute()` -- it reuses the
transport worked out in step 0.


In [ ]:
# Start Redis on the database node
!bash db_node_setup/run_on.sh database 'sudo bash -c "cd /root/SwarmAgents && docker compose up -d redis"'


## Trigger consensus from the database node

In [ ]:
!bash db_node_setup/run_on.sh database 'sudo bash -c "cd /root/SwarmAgents && ./batch_tests_v2.py --runs 1 --base-out run-h-30-100 --mode remote --agent-type resource --agents 30 --topology hierarchical --hierarchical-level1-agent-type resource --jobs 100 --db-host database --job-interval 120 --jobs-per-interval 1"'


In [ ]:
# Archive the results on the database node and pull them back here
!bash db_node_setup/run_on.sh database 'sudo bash -c "cd /root/SwarmAgents && tar -zcf /tmp/run-h-30-100.tgz run-h-30-100/"'
!bash db_node_setup/fetch.sh database /tmp/run-h-30-100.tgz run-h-30-100.tgz
!tar -zxf run-h-30-100.tgz && ls run-h-30-100/


**Parent Agents - LLM**

**Children Agents - Heuristic**

![Topolgy](./run-h-30-100/run01/hierarchical_topology.png)

![](./run-h-30-100/run01/latency_comparison_by_hierarchy_level.png)

### Delete the Slice

In [ ]:
# Delete BOTH slice parts
for sn in (slice_name_1, slice_name_2):
    try:
        s = fablib.get_slice(sn)
        s.delete()
        print(f"Deleted {sn}")
    except Exception as e:
        print(f"Could not delete {sn}: {e}")